# Fractional Factorial Analysis: 2^(4-1) Design

## Experimental Design Overview

This notebook analyzes experimental results from a **2^(4-1) half-fraction factorial design** with 8 runs.

### Factors (Independent Variables)
From `code/config/profiles/config_runX_openAI.yaml`:

| Factor | Config Parameter | Description | Levels |
|--------|-----------------|-------------|--------|
| **T1** | `use_openie` | OpenIE extraction | 0 (false), 1 (true) |
| **T2** | `use_enrichment_kb` | Knowledge base enrichment | 0 (false), 1 (true) |
| **Q1** | `use_shortcuts` | Query shortcuts | 0 (false), 1 (true) |
| **Q2** | `use_voting` + `expand_query_synonyms` | Voting & synonyms | 0 (both false), 1 (both true) |

### Design Matrix (from config comments)
```
Run | T1 | T2 | Q1 | Q2 | Binary
----|----|----|----|----|-------
  1 |  0 |  0 |  0 |  0 | 0000
  2 |  1 |  0 |  0 |  1 | 1001  
  3 |  0 |  1 |  0 |  1 | 0101
  4 |  1 |  1 |  0 |  0 | 1100
  5 |  0 |  0 |  1 |  1 | 0011
  6 |  1 |  0 |  1 |  0 | 1010
  7 |  0 |  1 |  1 |  0 | 0110
  8 |  1 |  1 |  1 |  1 | 1111
```

### Targets (Dependent Variables)
From `experiments/claude_constructed/results/`:
- `overall_accuracy`
- `accuracy_entailment`
- `accuracy_contradiction` (non-entailment)
- `accuracy_not_mentioned`
- `accuracy_uncertain`

### Fractional Factorial Properties
- **Resolution IV**: Main effects are clear of two-factor interactions
- **Defining relation**: T1*T2*Q1*Q2 = I (generator)
- **Aliasing**: Two-factor interactions confounded with other two-factor interactions

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import yaml
import warnings
warnings.filterwarnings('ignore')

# Statistical analysis
from scipy import stats
from itertools import combinations

# Optional imports
try:
    import statsmodels.api as sm
    from statsmodels.formula.api import ols
    from statsmodels.stats.anova import anova_lm
    STATSMODELS_AVAILABLE = True
except ImportError:
    STATSMODELS_AVAILABLE = False
    print("statsmodels not available. Install: pip install statsmodels")

plt.style.use('seaborn-v0_8-whitegrid')
print("Imports successful!")

## 1. Data Loading

Load features from config profiles and targets from result files.

In [ ]:
# Paths - adjust if running from different location
REPO_ROOT = Path("..")  # Assuming notebook is in data_analysis/
PROFILES_DIR = REPO_ROOT / "code" / "config" / "profiles"
RESULTS_DIR = REPO_ROOT / "experiments" / "claude_constructed" / "results"

print(f"Profiles directory: {PROFILES_DIR.resolve()}")
print(f"Results directory: {RESULTS_DIR.resolve()}")
print(f"Profiles exist: {PROFILES_DIR.exists()}")
print(f"Results exist: {RESULTS_DIR.exists()}")

In [ ]:
# Define the 2^(4-1) fractional factorial design matrix
# Based on config file comments: T1 T2 Q1 Q2
DESIGN_MATRIX = {
    1: {'T1': 0, 'T2': 0, 'Q1': 0, 'Q2': 0},  # 0000
    2: {'T1': 1, 'T2': 0, 'Q1': 0, 'Q2': 1},  # 1001
    3: {'T1': 0, 'T2': 1, 'Q1': 0, 'Q2': 1},  # 0101
    4: {'T1': 1, 'T2': 1, 'Q1': 0, 'Q2': 0},  # 1100
    5: {'T1': 0, 'T2': 0, 'Q1': 1, 'Q2': 1},  # 0011
    6: {'T1': 1, 'T2': 0, 'Q1': 1, 'Q2': 0},  # 1010
    7: {'T1': 0, 'T2': 1, 'Q1': 1, 'Q2': 0},  # 0110
    8: {'T1': 1, 'T2': 1, 'Q1': 1, 'Q2': 1},  # 1111
}

# Factor descriptions
FACTOR_INFO = {
    'T1': {'name': 'OpenIE', 'config_key': 'use_openie'},
    'T2': {'name': 'KB Enrichment', 'config_key': 'use_enrichment_kb'},
    'Q1': {'name': 'Shortcuts', 'config_key': 'use_shortcuts'},
    'Q2': {'name': 'Voting+Synonyms', 'config_key': 'use_voting'},
}

FACTORS = ['T1', 'T2', 'Q1', 'Q2']

# Display design matrix
design_df = pd.DataFrame(DESIGN_MATRIX).T
design_df.index.name = 'Run'
print("2^(4-1) Fractional Factorial Design Matrix:")
display(design_df)

In [ ]:
def load_config_features(profiles_dir: Path) -> pd.DataFrame:
    """
    Load feature flags from YAML config profiles.
    """
    data = []
    
    for run_num in range(1, 9):
        config_path = profiles_dir / f"config_run{run_num}_openAI.yaml"
        
        if config_path.exists():
            with open(config_path, 'r') as f:
                config = yaml.safe_load(f)
            
            features = config.get('features', {})
            
            row = {
                'run': run_num,
                'T1': int(features.get('use_openie', False)),
                'T2': int(features.get('use_enrichment_kb', False)),
                'Q1': int(features.get('use_shortcuts', False)),
                'Q2': int(features.get('use_voting', False)),  # Q2 includes both voting and synonyms
            }
            data.append(row)
        else:
            print(f"Warning: Config not found: {config_path}")
    
    return pd.DataFrame(data)

def load_results(results_dir: Path) -> pd.DataFrame:
    """
    Load accuracy results from JSON result files.
    """
    data = []
    
    for result_file in sorted(results_dir.glob("config_run*_openAI_*.json")):
        # Extract run number from filename
        filename = result_file.name
        run_num = int(filename.split('_')[1].replace('run', ''))
        
        with open(result_file, 'r') as f:
            result = json.load(f)
        
        metadata = result.get('metadata', {})
        per_label = metadata.get('per_label_accuracy', {})
        
        row = {
            'run': run_num,
            'overall_accuracy': metadata.get('overall_accuracy', np.nan),
            'accuracy_entailment': per_label.get('entailment', {}).get('accuracy', np.nan),
            'accuracy_contradiction': per_label.get('contradiction', {}).get('accuracy', np.nan),
            'accuracy_not_mentioned': per_label.get('not_mentioned', {}).get('accuracy', np.nan),
            'accuracy_uncertain': per_label.get('uncertain', {}).get('accuracy', np.nan),
            'total_correct': metadata.get('total_correct', np.nan),
            'total_evaluated': metadata.get('total_evaluated', np.nan),
        }
        data.append(row)
    
    return pd.DataFrame(data)

# Load data
df_features = load_config_features(PROFILES_DIR)
df_results = load_results(RESULTS_DIR)

print(f"Loaded {len(df_features)} config profiles")
print(f"Loaded {len(df_results)} result files")

In [ ]:
# Merge features and results
df = pd.merge(df_features, df_results, on='run', how='outer')
df = df.sort_values('run').reset_index(drop=True)

# Define target columns
TARGET_COLS = [
    'overall_accuracy',
    'accuracy_entailment',
    'accuracy_contradiction',
    'accuracy_not_mentioned',
    'accuracy_uncertain'
]

print("\n" + "="*70)
print("COMPLETE EXPERIMENTAL DATA")
print("="*70)
display(df)

In [ ]:
# Verify design matrix matches expected fractional factorial
print("\n" + "="*70)
print("DESIGN VERIFICATION")
print("="*70)

print("\nExpected design (from config comments):")
for run, factors in DESIGN_MATRIX.items():
    binary = ''.join(str(factors[f]) for f in ['T1', 'T2', 'Q1', 'Q2'])
    print(f"  Run {run}: {binary}")

print("\nActual design (from loaded configs):")
for _, row in df.iterrows():
    binary = ''.join(str(int(row[f])) for f in FACTORS)
    print(f"  Run {int(row['run'])}: {binary}")

# Check if generator relation holds: T1*T2*Q1*Q2 = I (product should be constant)
df['generator'] = df['T1'] * df['T2'] * df['Q1'] * df['Q2']
# In coded form (-1, +1), the product should be +1 for all runs in a half-fraction
# In (0,1) form, we need to convert
df['T1_coded'] = 2*df['T1'] - 1
df['T2_coded'] = 2*df['T2'] - 1
df['Q1_coded'] = 2*df['Q1'] - 1
df['Q2_coded'] = 2*df['Q2'] - 1
df['generator_coded'] = df['T1_coded'] * df['T2_coded'] * df['Q1_coded'] * df['Q2_coded']

print(f"\nGenerator (T1*T2*Q1*Q2 in coded form):")
print(f"  Values: {df['generator_coded'].tolist()}")
if df['generator_coded'].nunique() == 1:
    print(f"  ✓ Valid half-fraction: all products = {df['generator_coded'].iloc[0]}")
else:
    print(f"  ✗ Warning: generator not constant - design may not be balanced")

## 2. Descriptive Statistics

In [ ]:
print("="*70)
print("DESCRIPTIVE STATISTICS")
print("="*70)

print("\n--- Target Variable Statistics ---")
display(df[TARGET_COLS].describe().round(4))

print("\n--- Accuracy by Run ---")
display(df[['run'] + FACTORS + TARGET_COLS].round(4))

In [ ]:
# Correlation matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Factor-Target correlations
corr_ft = df[FACTORS + TARGET_COLS].corr()
mask_ft = np.zeros_like(corr_ft.loc[FACTORS, TARGET_COLS])
sns.heatmap(
    corr_ft.loc[FACTORS, TARGET_COLS], 
    annot=True, 
    cmap='RdBu_r', 
    center=0,
    ax=axes[0],
    vmin=-1, vmax=1,
    fmt='.3f'
)
axes[0].set_title('Factor-Target Correlations', fontsize=12)

# Target-Target correlations
corr_tt = df[TARGET_COLS].corr()
sns.heatmap(
    corr_tt, 
    annot=True, 
    cmap='RdBu_r', 
    center=0,
    ax=axes[1],
    vmin=-1, vmax=1,
    fmt='.3f'
)
axes[1].set_title('Target-Target Correlations', fontsize=12)

plt.tight_layout()
plt.savefig('correlation_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Main Effects Analysis

For a 2-level factorial design:
$$\text{Main Effect}_i = \bar{Y}_{i+} - \bar{Y}_{i-}$$

where $\bar{Y}_{i+}$ is the mean response when factor $i$ is at high level (+1) and $\bar{Y}_{i-}$ is the mean when at low level (-1 or 0).

In [ ]:
def compute_main_effects(df, factors, target):
    """
    Compute main effects for each factor.
    Main effect = mean(high) - mean(low)
    """
    effects = {}
    
    for factor in factors:
        mean_low = df[df[factor] == 0][target].mean()
        mean_high = df[df[factor] == 1][target].mean()
        effects[factor] = mean_high - mean_low
    
    return effects

# Compute main effects for all targets
print("="*70)
print("MAIN EFFECTS ANALYSIS")
print("="*70)

main_effects_df = pd.DataFrame()

for target in TARGET_COLS:
    effects = compute_main_effects(df, FACTORS, target)
    main_effects_df[target] = pd.Series(effects)

# Add factor descriptions
main_effects_df.index = [f"{f} ({FACTOR_INFO[f]['name']})" for f in FACTORS]

print("\nMain Effects (High - Low):")
display(main_effects_df.round(4))

print("\nInterpretation:")
print("  Positive effect: Factor increases accuracy when turned ON")
print("  Negative effect: Factor decreases accuracy when turned ON")

In [ ]:
# Visualize main effects
fig, axes = plt.subplots(1, len(TARGET_COLS), figsize=(4*len(TARGET_COLS), 5))

for i, target in enumerate(TARGET_COLS):
    effects = main_effects_df[target]
    colors = ['#2ecc71' if e > 0 else '#e74c3c' for e in effects]
    
    y_pos = np.arange(len(effects))
    bars = axes[i].barh(y_pos, effects.values, color=colors, alpha=0.8, edgecolor='black')
    axes[i].axvline(x=0, color='black', linestyle='-', linewidth=1)
    axes[i].set_yticks(y_pos)
    axes[i].set_yticklabels([f.split(' (')[0] for f in effects.index])
    axes[i].set_xlabel('Effect Size')
    axes[i].set_title(f'{target.replace("accuracy_", "").replace("_", " ").title()}', fontsize=10)
    
    # Add value labels
    for bar, val in zip(bars, effects.values):
        x_pos = val + 0.005 if val >= 0 else val - 0.005
        ha = 'left' if val >= 0 else 'right'
        axes[i].text(x_pos, bar.get_y() + bar.get_height()/2, 
                    f'{val:.3f}', va='center', ha=ha, fontsize=9)

plt.suptitle('Main Effects by Target Variable', y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig('main_effects.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Main effects plot (means at each level)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

target = 'overall_accuracy'

for i, factor in enumerate(FACTORS):
    ax = axes[i]
    
    # Calculate means at each level
    means = df.groupby(factor)[target].mean()
    stds = df.groupby(factor)[target].std()
    
    x = [0, 1]
    ax.errorbar(x, means.values, yerr=stds.values, fmt='o-', capsize=5, 
                markersize=10, linewidth=2, color='#3498db')
    
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['OFF (0)', 'ON (1)'])
    ax.set_xlabel(f'{factor} ({FACTOR_INFO[factor]["name"]})')
    ax.set_ylabel('Overall Accuracy')
    ax.set_title(f'Main Effect of {factor}')
    
    # Add effect size annotation
    effect = means[1] - means[0]
    ax.annotate(f'Effect: {effect:+.4f}', xy=(0.5, means.mean()), 
                fontsize=11, ha='center', 
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle(f'Main Effects Plots for Overall Accuracy', y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig('main_effects_plots.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Two-Factor Interaction Effects

For a 2-level design, the interaction effect is:
$$\text{Interaction}_{ij} = \frac{1}{2}[(\bar{Y}_{i+j+} - \bar{Y}_{i-j+}) - (\bar{Y}_{i+j-} - \bar{Y}_{i-j-})]$$

### Aliasing in 2^(4-1) Design
With generator I = T1*T2*Q1*Q2, the aliasing structure is:
- T1 = T2*Q1*Q2
- T2 = T1*Q1*Q2  
- Q1 = T1*T2*Q2
- Q2 = T1*T2*Q1
- T1*T2 = Q1*Q2
- T1*Q1 = T2*Q2
- T1*Q2 = T2*Q1

In [ ]:
def compute_interaction_effects(df, factors, target):
    """
    Compute two-way interaction effects.
    Uses coded variables (-1, +1).
    """
    interactions = {}
    
    for f1, f2 in combinations(factors, 2):
        # Create interaction column (in coded form)
        coded_f1 = 2*df[f1] - 1
        coded_f2 = 2*df[f2] - 1
        interaction = coded_f1 * coded_f2
        
        # Effect is half the difference between high and low interaction
        mean_high = df[interaction == 1][target].mean()
        mean_low = df[interaction == -1][target].mean()
        
        interactions[f"{f1}:{f2}"] = (mean_high - mean_low) / 2
    
    return interactions

# Compute interactions for all targets
print("="*70)
print("TWO-FACTOR INTERACTION EFFECTS")
print("="*70)

interactions_df = pd.DataFrame()

for target in TARGET_COLS:
    interactions = compute_interaction_effects(df, FACTORS, target)
    interactions_df[target] = pd.Series(interactions)

print("\nTwo-Way Interaction Effects:")
display(interactions_df.round(4))

print("\n--- Aliasing Structure (Resolution IV) ---")
print("  T1:T2 is aliased with Q1:Q2")
print("  T1:Q1 is aliased with T2:Q2")
print("  T1:Q2 is aliased with T2:Q1")

In [ ]:
# Interaction plots for overall accuracy
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

target = 'overall_accuracy'
interaction_pairs = list(combinations(FACTORS, 2))

for idx, (f1, f2) in enumerate(interaction_pairs):
    ax = axes[idx]
    
    # Calculate means for each combination
    for level_f2, style, color in [(0, '--', '#e74c3c'), (1, '-', '#2ecc71')]:
        subset = df[df[f2] == level_f2]
        means = subset.groupby(f1)[target].mean()
        
        label = f'{f2}={level_f2} ({"OFF" if level_f2==0 else "ON"})'
        ax.plot([0, 1], means.values, marker='o', linestyle=style, 
                color=color, linewidth=2, markersize=8, label=label)
    
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['OFF', 'ON'])
    ax.set_xlabel(f'{f1} ({FACTOR_INFO[f1]["name"]})')
    ax.set_ylabel(target)
    ax.set_title(f'{f1} x {f2} Interaction')
    ax.legend(title=f2, loc='best')
    
    # Add interaction effect
    effect = interactions_df.loc[f"{f1}:{f2}", target]
    ax.annotate(f'Int: {effect:.4f}', xy=(0.5, ax.get_ylim()[0]), 
                xytext=(0.5, ax.get_ylim()[0] + 0.02),
                fontsize=9, ha='center')

plt.suptitle(f'Two-Factor Interaction Plots for Overall Accuracy', y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig('interaction_plots.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Linear Regression Model with Effects

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

def fit_factorial_model(df, factors, target):
    """
    Fit a linear model with main effects and interactions.
    Uses coded variables (-1, +1) for interpretable coefficients.
    """
    # Create coded design matrix
    X_coded = pd.DataFrame()
    for f in factors:
        X_coded[f] = 2*df[f] - 1  # Convert 0/1 to -1/+1
    
    # Add interactions
    for f1, f2 in combinations(factors, 2):
        X_coded[f"{f1}:{f2}"] = X_coded[f1] * X_coded[f2]
    
    y = df[target].values
    
    # Fit model
    model = LinearRegression()
    model.fit(X_coded, y)
    
    # R-squared
    r2 = model.score(X_coded, y)
    
    # Coefficients (divide by 2 to get effects in standard DOE notation)
    coef_df = pd.DataFrame({
        'Term': X_coded.columns.tolist(),
        'Coefficient': model.coef_,
        'Effect': model.coef_ * 2  # Full effect = 2 * coefficient for coded vars
    })
    
    return {
        'model': model,
        'r2': r2,
        'intercept': model.intercept_,
        'coefficients': coef_df.sort_values('Effect', key=abs, ascending=False)
    }

# Fit models for all targets
print("="*70)
print("LINEAR REGRESSION MODELS (Coded Variables)")
print("="*70)

models = {}
for target in TARGET_COLS:
    result = fit_factorial_model(df, FACTORS, target)
    models[target] = result
    
    print(f"\n--- {target} ---")
    print(f"R² = {result['r2']:.4f}")
    print(f"Intercept (Grand Mean) = {result['intercept']:.4f}")
    print("\nCoefficients (sorted by |Effect|):")
    display(result['coefficients'].round(4))

In [ ]:
# Pareto chart of effects for overall accuracy
target = 'overall_accuracy'
coef_df = models[target]['coefficients'].copy()
coef_df['abs_effect'] = coef_df['Effect'].abs()
coef_df = coef_df.sort_values('abs_effect', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#2ecc71' if e > 0 else '#e74c3c' for e in coef_df['Effect']]
bars = ax.barh(coef_df['Term'], coef_df['Effect'], color=colors, alpha=0.8, edgecolor='black')
ax.axvline(x=0, color='black', linestyle='-', linewidth=1)
ax.set_xlabel('Effect')
ax.set_ylabel('Term')
ax.set_title(f'Pareto Chart of Effects: {target}\n(Green = Positive, Red = Negative)')

# Add value labels
for bar, val in zip(bars, coef_df['Effect']):
    x_pos = val + 0.005 if val >= 0 else val - 0.005
    ha = 'left' if val >= 0 else 'right'
    ax.text(x_pos, bar.get_y() + bar.get_height()/2, 
            f'{val:.4f}', va='center', ha=ha, fontsize=9)

plt.tight_layout()
plt.savefig('pareto_effects.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. ANOVA Analysis (if statsmodels available)

In [ ]:
if STATSMODELS_AVAILABLE:
    print("="*70)
    print("ANOVA ANALYSIS")
    print("="*70)
    
    # Use coded variables for ANOVA
    df_anova = df.copy()
    for f in FACTORS:
        df_anova[f] = 2*df_anova[f] - 1
    
    target = 'overall_accuracy'
    
    # Full model with main effects and 2-way interactions
    formula = f"{target} ~ T1 + T2 + Q1 + Q2 + T1:T2 + T1:Q1 + T1:Q2 + T2:Q1 + T2:Q2 + Q1:Q2"
    
    print(f"\nFormula: {formula}")
    
    model = ols(formula, data=df_anova).fit()
    
    print("\n--- Model Summary ---")
    print(model.summary())
    
    print("\n--- ANOVA Table (Type II) ---")
    anova_table = anova_lm(model, typ=2)
    display(anova_table.round(4))
else:
    print("ANOVA requires statsmodels. Install with: pip install statsmodels")

## 7. Per-Class Accuracy Analysis

In [ ]:
# Compare accuracies across classes
class_targets = [t for t in TARGET_COLS if t != 'overall_accuracy']

print("="*70)
print("PER-CLASS ACCURACY ANALYSIS")
print("="*70)

# Summary statistics
print("\n--- Class Accuracy Statistics ---")
class_stats = df[class_targets].describe().T
class_stats['range'] = class_stats['max'] - class_stats['min']
display(class_stats.round(4))

In [ ]:
# Box plot comparison
fig, ax = plt.subplots(figsize=(12, 6))

# Melt for plotting
df_melted = df.melt(
    id_vars=['run'] + FACTORS,
    value_vars=class_targets,
    var_name='Class',
    value_name='Accuracy'
)
df_melted['Class'] = df_melted['Class'].str.replace('accuracy_', '').str.replace('_', ' ').str.title()

palette = {'Entailment': '#2ecc71', 'Contradiction': '#e74c3c', 
           'Not Mentioned': '#3498db', 'Uncertain': '#f39c12'}

sns.boxplot(data=df_melted, x='Class', y='Accuracy', palette=palette, ax=ax)
sns.stripplot(data=df_melted, x='Class', y='Accuracy', color='black', alpha=0.5, size=6, ax=ax)

ax.set_xlabel('NLI Class')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy Distribution by NLI Class')

plt.tight_layout()
plt.savefig('class_accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Main effects comparison across classes
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, factor in enumerate(FACTORS):
    ax = axes[idx]
    
    # Calculate effect for each class
    effects = {}
    for target in class_targets:
        mean_low = df[df[factor] == 0][target].mean()
        mean_high = df[df[factor] == 1][target].mean()
        class_name = target.replace('accuracy_', '').replace('_', ' ').title()
        effects[class_name] = mean_high - mean_low
    
    colors = [palette.get(c, 'gray') for c in effects.keys()]
    bars = ax.bar(effects.keys(), effects.values(), color=colors, alpha=0.8, edgecolor='black')
    ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
    ax.set_ylabel('Effect on Accuracy')
    ax.set_title(f'Effect of {factor} ({FACTOR_INFO[factor]["name"]}) by Class')
    ax.tick_params(axis='x', rotation=45)
    
    # Add value labels
    for bar, val in zip(bars, effects.values()):
        y_pos = val + 0.01 if val >= 0 else val - 0.01
        va = 'bottom' if val >= 0 else 'top'
        ax.text(bar.get_x() + bar.get_width()/2, y_pos, f'{val:.3f}', 
                ha='center', va=va, fontsize=9)

plt.suptitle('Factor Effects by NLI Class', y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig('factor_effects_by_class.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Optimal Configuration Analysis

In [ ]:
print("="*70)
print("OPTIMAL CONFIGURATIONS")
print("="*70)

for target in TARGET_COLS:
    best_idx = df[target].idxmax()
    best_row = df.loc[best_idx]
    
    print(f"\n--- Best for {target} ---")
    print(f"  Run: {int(best_row['run'])}")
    print(f"  Accuracy: {best_row[target]:.4f}")
    print(f"  Configuration:")
    for f in FACTORS:
        val = int(best_row[f])
        print(f"    {f} ({FACTOR_INFO[f]['name']}): {'ON' if val else 'OFF'}")

In [ ]:
# Heatmap of all runs vs targets
fig, ax = plt.subplots(figsize=(12, 8))

# Create labels with factor settings
run_labels = []
for _, row in df.iterrows():
    factors_str = ''.join([str(int(row[f])) for f in FACTORS])
    run_labels.append(f"Run {int(row['run'])} ({factors_str})")

heatmap_data = df[TARGET_COLS].copy()
heatmap_data.index = run_labels
heatmap_data.columns = [c.replace('accuracy_', '').replace('_', ' ').title() for c in TARGET_COLS]

sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='RdYlGn', 
            center=heatmap_data.values.mean(), ax=ax,
            cbar_kws={'label': 'Accuracy'})
ax.set_xlabel('Target Variable')
ax.set_ylabel('Run (T1 T2 Q1 Q2)')
ax.set_title('Accuracy Heatmap: All Runs vs All Targets')

plt.tight_layout()
plt.savefig('accuracy_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Summary and Conclusions

In [ ]:
print("="*70)
print("ANALYSIS SUMMARY")
print("="*70)

# Overall statistics
print(f"""
EXPERIMENTAL DESIGN
-------------------
Design Type: 2^(4-1) Half-Fraction Factorial
Resolution: IV (main effects clear of 2FI)
Number of Runs: {len(df)}
Number of Factors: {len(FACTORS)}
Generator: I = T1*T2*Q1*Q2
""")

# Most important effects for overall accuracy
target = 'overall_accuracy'
main_eff = compute_main_effects(df, FACTORS, target)
sorted_effects = sorted(main_eff.items(), key=lambda x: abs(x[1]), reverse=True)

print(f"""
KEY FINDINGS FOR OVERALL ACCURACY
----------------------------------
Mean accuracy: {df[target].mean():.4f}
Std deviation: {df[target].std():.4f}
Range: [{df[target].min():.4f}, {df[target].max():.4f}]
""")

print("Main Effects (ranked by magnitude):")
for factor, effect in sorted_effects:
    direction = "+" if effect > 0 else "-"
    impact = "increases" if effect > 0 else "decreases"
    print(f"  {factor} ({FACTOR_INFO[factor]['name']}): {effect:+.4f} -- Turning ON {impact} accuracy")

# Best configuration
best_run = df.loc[df[target].idxmax()]
print(f"""
RECOMMENDED CONFIGURATION
-------------------------
Based on overall accuracy optimization:
  Best Run: {int(best_run['run'])}
  Accuracy: {best_run[target]:.4f}
  Settings:""")
for f in FACTORS:
    print(f"    {f} ({FACTOR_INFO[f]['name']}): {'ON' if best_run[f] else 'OFF'}")

In [ ]:
# Export results to CSV
print("\n" + "="*70)
print("EXPORTING RESULTS")
print("="*70)

# Main data
df.to_csv('experiment_data.csv', index=False)
print("Saved: experiment_data.csv")

# Main effects
main_effects_df.to_csv('main_effects.csv')
print("Saved: main_effects.csv")

# Interaction effects
interactions_df.to_csv('interaction_effects.csv')
print("Saved: interaction_effects.csv")

print("\nAll analysis complete!")